# NB14 — Final Audit and Downloadable ZIP | NIR-HUEVOS 2026

Run this notebook only after NB11, NB12, and NB13 report `SUCCESS`. It creates a self-auditing ZIP in Google Drive and then starts a browser download. The archive contains the new notebooks, final result files, figures, protocols, execution-status records, and frozen split definitions; it does **not** duplicate the raw dataset.


In [1]:
from google.colab import drive, files
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, shutil, subprocess, sys, zipfile
import numpy as np
import pandas as pd

PROJECT_ROOT=Path('/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026')
NOTEBOOK_DIR=PROJECT_ROOT/'04_NOTEBOOKS'/'REVISION_REVIEWERS_2026_09'
RESULT_ROOT=PROJECT_ROOT/'05_RESULTS'/'REVISION_REVIEWERS_2026_09'
FIG_ROOT=PROJECT_ROOT/'06_FIGURES'/'REVISION_REVIEWERS_2026_09'
SPLIT_DIR=PROJECT_ROOT/'03_SPLITS_FROZEN'
ZIP_DIR=PROJECT_ROOT/'05_RESULTS'/'ZIP_PACKAGES'
PACKAGE_DIR=RESULT_ROOT/'NB14_FINAL_PACKAGE_STAGING'
ZIP_PATH=ZIP_DIR/'NIR_HUEVOS_REVIEWER_ROUND_2026_09.zip'
ZIP_DIR.mkdir(parents=True,exist_ok=True)

required={
 'NB11':RESULT_ROOT/'NB11_CNN1D_REVIEWER_BENCHMARK'/'EXECUTION_STATUS.json',
 'NB12':RESULT_ROOT/'NB12_CLUSTERED_UNCERTAINTY_AND_INFLUENCE'/'EXECUTION_STATUS.json',
 'NB13':RESULT_ROOT/'NB13_REVIEWER_TABLES_FIGURES'/'EXECUTION_STATUS.json'
}
status_rows=[]
for stage,path in required.items():
    assert path.exists(),f'Missing {path}'
    s=json.loads(path.read_text())
    assert s['status']=='SUCCESS',f'{stage} is not complete: {s}'
    status_rows.append({'stage':stage,'status':s['status'],'completed_at_utc':s.get('completed_at_utc')})
display(pd.DataFrame(status_rows))


In [ ]:
# Assemble a clean package from completed artifacts
if PACKAGE_DIR.exists(): shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True)

def copy_tree_selected(src,dst,exclude_names=()):
    dst.mkdir(parents=True,exist_ok=True)
    for p in sorted(src.rglob('*')):
        if not p.is_file(): continue
        if any(part in exclude_names for part in p.parts): continue
        rel=p.relative_to(src); target=dst/rel; target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(p,target)

copy_tree_selected(NOTEBOOK_DIR,PACKAGE_DIR/'04_NOTEBOOKS')
copy_tree_selected(RESULT_ROOT,PACKAGE_DIR/'05_RESULTS',exclude_names=('_CHECKPOINT','NB14_FINAL_PACKAGE_STAGING'))
copy_tree_selected(FIG_ROOT,PACKAGE_DIR/'06_FIGURES')
for p in sorted(SPLIT_DIR.glob('*')):
    if p.is_file():
        (PACKAGE_DIR/'03_SPLITS_FROZEN').mkdir(exist_ok=True)
        shutil.copy2(p,PACKAGE_DIR/'03_SPLITS_FROZEN'/p.name)

readme="""NIR-HUEVOS — ADDITIONAL REVIEWER ANALYSES (September 2026)

Execution order:
1. NB11_CNN1D_REVIEWER_BENCHMARK.ipynb
2. NB12_CLUSTERED_UNCERTAINTY_AND_INFLUENCE.ipynb
3. NB13_REVIEWER_TABLES_FIGURES.ipynb
4. NB14_FINAL_RESULTS_PACKAGE.ipynb

Scope:
- NB01–NB10 primary results remain unchanged.
- NB11 is an additional compact 1D-CNN comparator using exact frozen egg-disjoint splits.
- NB12 performs inference at the egg level (N=30), including 10,000 cluster bootstrap replicates.
- NB13 generates reviewer-facing tables and figures.
- The raw dataset is not duplicated in this archive. Its SHA-256 is recorded in the NB11 protocol.
- External validation is still required before deployment claims.
"""
(PACKAGE_DIR/'README.txt').write_text(readme,encoding='utf-8')


In [ ]:
# SHA-256 manifest, inventory, and environment snapshot
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''): h.update(b)
    return h.hexdigest()

manifest=[]
for p in sorted(PACKAGE_DIR.rglob('*')):
    if p.is_file():
        manifest.append({'relative_path':p.relative_to(PACKAGE_DIR).as_posix(),'size_bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(PACKAGE_DIR/'MANIFEST_SHA256.csv',index=False)
snapshot={
 'created_at_utc':datetime.now(timezone.utc).isoformat(),
 'python':sys.version,'platform':platform.platform(),
 'numpy':np.__version__,'pandas':pd.__version__,
 'stages':status_rows,'file_count_before_manifest':len(manifest),
 'raw_dataset_included':False,'primary_NB01_NB10_modified':False
}
(PACKAGE_DIR/'PACKAGE_METADATA.json').write_text(json.dumps(snapshot,indent=2),encoding='utf-8')
print('Package files:',len(list(PACKAGE_DIR.rglob('*'))))


In [ ]:
# Build and verify ZIP; saved in Drive automatically
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(PACKAGE_DIR.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(PACKAGE_DIR).as_posix())
with zipfile.ZipFile(ZIP_PATH,'r') as z:
    assert z.testzip() is None
    members=z.namelist()
    assert 'README.txt' in members and 'MANIFEST_SHA256.csv' in members and 'PACKAGE_METADATA.json' in members

zip_sha=sha256_file(ZIP_PATH)
final_status={
 'notebook':'NB14_FINAL_RESULTS_PACKAGE.ipynb','status':'SUCCESS',
 'completed_at_utc':datetime.now(timezone.utc).isoformat(),
 'zip_path':str(ZIP_PATH),'zip_size_bytes':ZIP_PATH.stat().st_size,
 'zip_sha256':zip_sha,'zip_member_count':len(members),'zip_integrity_test':'PASS'
}
(RESULT_ROOT/'NB14_EXECUTION_STATUS.json').write_text(json.dumps(final_status,indent=2),encoding='utf-8')
print(json.dumps(final_status,indent=2))


## Download

The next cell downloads the exact ZIP that remains stored in Drive. If the browser blocks the first download, allow downloads for Colab and run only that final cell again.


In [ ]:
files.download(str(ZIP_PATH))
